# Exploratory Data Analysis 2.0

> Goal: Filter and examine team-level data with complete match information for model exploration.

## 1) Load, Filter, and Preview Raw Data

In [1]:
import pandas as pd

# Load raw match-level and team-level data.
df = pd.read_csv('../data/raw/2026_LoL_OraclesElixir.csv', low_memory=False)

# Filter for complete match records only (exclude partial/incomplete data).
df = df[df['datacompleteness'] == 'complete']

# Filter for team-level rows only (exclude player-level rows).
df = df[df["position"] == 'team']

# Standardize side column to lowercase for consistency.
df['side'] = df['side'].str.lower()

# Quick preview: show game ID distribution across teams.
df["gameid"].value_counts().head()

gameid
LOLTMNT05_171038    2
LOLTMNT05_172024    2
LOLTMNT05_171043    2
LOLTMNT05_171051    2
LOLTMNT05_171066    2
Name: count, dtype: int64

## 2) Inspect Data Profile

In [2]:
# Display the filtered dataframe to inspect team-level data structure and sample rows.
df[df["position"] == 'team']

,gameid,datacompleteness,url,league,year,split,playoffs,date,game,patch,...,opp_csat25,golddiffat25,xpdiffat25,csdiffat25,killsat25,assistsat25,deathsat25,opp_killsat25,opp_assistsat25,opp_deathsat25
10,LOLTMNT05_171038,complete,NaN,LIT,2026,Winter,0,2026-01-08 17:08:27,1,16.01,...,824.0,1755.0,-1013.0,-29.0,20.0,22.0,20.0,20.0,26.0,20.0
11,LOLTMNT05_171038,complete,NaN,LIT,2026,Winter,0,2026-01-08 17:08:27,1,16.01,...,795.0,-1755.0,1013.0,29.0,20.0,26.0,20.0,20.0,22.0,20.0
22,LOLTMNT05_172024,complete,NaN,LIT,2026,Winter,0,2026-01-08 17:56:21,1,16.01,...,906.0,99.0,-797.0,80.0,3.0,4.0,9.0,9.0,15.0,3.0
23,LOLTMNT05_172024,complete,NaN,LIT,2026,Winter,0,2026-01-08 17:56:21,1,16.01,...,986.0,-99.0,797.0,-80.0,9.0,15.0,3.0,3.0,4.0,9.0
34,LOLTMNT05_171043,complete,NaN,LIT,2026,Winter,0,2026-01-08 18:51:06,1,16.01,...,871.0,-1097.0,-6230.0,-5.0,8.0,12.0,14.0,14.0,26.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60995,LOLTMNT02_411607,complete,NaN,LCS,2026,Spring,1,2026-05-24 22:03:57,3,16.10,...,906.0,776.0,-4931.0,24.0,11.0,24.0,16.0,16.0,31.0,11.0
61006,LOLTMNT02_411614,complete,NaN,LCS,2026,Spring,1,2026-05-24 22:54:16,4,16.10,...,975.0,-5229.0,-314.0,-54.0,2.0,7.0,6.0,6.0,11.0,2.0
61007,LOLTMNT02_411614,complete,NaN,LCS,2026,Spring,1,2026-05-24 22:54:16,4,16.10,...,921.0,5229.0,314.0,54.0,6.0,11.0,2.0,2.0,7.0,6.0
61018,LOLTMNT02_411619,complete,NaN,LCS,2026,Spring,1,2026-05-24 23:44:08,5,16.10,...,971.0,1335.0,-2624.0,27.0,1.0,1.0,2.0,2.0,4.0,1.0


## 3) Build Match-Level Dataset

In [3]:
# Select only fields needed to build one row per game (blue side vs red side).
core_columns = ["gameid", "date", "patch", "league", "side", "teamname", "result"]
team_df = df[core_columns].copy()

# Collect transformed match rows here.
matches = []

for gameid, group in team_df.groupby("gameid"):
    # Keep only complete games with exactly two team rows.
    if len(group) != 2:
        continue

    blue = group[group["side"] == "blue"]
    red = group[group["side"] == "red"]

    # Skip malformed groups missing one side.
    if blue.empty or red.empty:
        continue

    blue_row = blue.iloc[0]
    red_row = red.iloc[0]

    matches.append(
        {
            "gameid": gameid,
            "date": blue_row["date"],
            "patch": blue_row["patch"],
            "league": blue_row["league"],
            "blue_team": blue_row["teamname"],
            "red_team": red_row["teamname"],
            # Oracle's Elixir result is numeric: 1 = win, 0 = loss.
            "blue_side_win": int(blue_row["result"]),
        }
    )

matches_df = pd.DataFrame(matches)

## 4) Validate Match Dataset

Run sanity checks to confirm schema, class balance, missingness, and team-name consistency before export.

In [4]:
# Confirm column dtypes and non-null counts after transformation.
matches_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4682 entries, 0 to 4681
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   gameid         4682 non-null   str    
 1   date           4682 non-null   str    
 2   patch          4682 non-null   float64
 3   league         4682 non-null   str    
 4   blue_team      4682 non-null   str    
 5   red_team       4682 non-null   str    
 6   blue_side_win  4682 non-null   int64  
dtypes: float64(1), int64(1), str(5)
memory usage: 256.2 KB


In [5]:
# Check class distribution for target balance (blue-side wins vs losses).
matches_df["blue_side_win"].value_counts()

blue_side_win
1    2465
0    2217
Name: count, dtype: int64

In [6]:
# Verify no missing values in final modeling fields.
matches_df.isnull().sum()

gameid           0
date             0
patch            0
league           0
blue_team        0
red_team         0
blue_side_win    0
dtype: int64

In [7]:
# Trim whitespace in team names to prevent duplicate labels caused by spacing.
matches_df["blue_team"] = matches_df["blue_team"].str.strip()
matches_df["red_team"] = matches_df["red_team"].str.strip()

In [8]:
# Re-check the cleaned dataset sample.
matches_df.head()

,gameid,date,patch,league,blue_team,red_team,blue_side_win
0,LOLTMNT01_318843,2026-01-13 19:08:52,16.01,ROL,The Bandits,Dynasty,1
1,LOLTMNT01_318863,2026-01-13 20:25:15,16.01,ROL,mCon esports,Myth Esports,1
2,LOLTMNT01_318864,2026-01-13 20:25:26,16.01,ROL,Once Upon A Team,Frites Esports Club,0
3,LOLTMNT01_318874,2026-01-13 21:22:16,16.01,ROL,mCon esports,Frites Esports Club,0
4,LOLTMNT01_318875,2026-01-13 21:20:31,16.01,ROL,Senshi eSports,The Bandits,0


In [9]:
# Confirm number of rows and columns in the final dataset.
matches_df.shape

(4682, 7)

In [10]:
# Compare unique team counts by side as a quick consistency check.
matches_df["blue_team"].nunique(), matches_df["red_team"].nunique()

(317, 320)

## 5) Export Processed Dataset

Persist the cleaned match-level table so training scripts can consume a stable file from data/processed.

In [11]:
# Save final match-level dataset for downstream modeling.
matches_df.to_csv('../data/processed/processed_matches.csv', index=False)